## OPTIMAL EXECUTION

The Almgren-Chriss (2000) model of optimal execution, in discrete time: the
closed-form liquidation schedule that trades off market impact against timing
risk.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Standard library imports

# Third-party imports
import numpy as np
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

# First-party imports
from xpectral.quant.execution import (
    cost_variance,
    decay_parameter,
    efficient_frontier,
    expected_cost,
    optimal_holdings,
)

## Inputs

Every input is anchored to the stock itself rather than stated as an abstract
dollar quantity: price and a percentage daily volatility combine into the
dollar volatility $\sigma$ the model needs, and a target notional combines
with price into a share count $X$. $\eta$ and $\gamma$ (impact coefficients)
and $T$ (the trading horizon) are stated directly, and the horizon is split
into $N$ trading steps of length $\tau = T / N$. Three risk-aversion levels
$\lambda$ are chosen to illustrate risk-neutral, moderate, and aggressive
execution; sweeping $\lambda$ continuously later traces the full efficient
frontier.

Each $\lambda$ maps to a decay rate $\kappa$ that solves
$\frac{2}{\tau^2}(\cosh(\kappa\tau) - 1) = \tilde\kappa^2$, with
$\tilde\kappa^2 = \lambda\sigma^2 / \tilde\eta$ and
$\tilde\eta = \eta - \frac{1}{2}\gamma\tau$.

In [3]:
price = 50.0  # $/share
sigma_pct = 0.01  # daily volatility, as a fraction of price
sigma = sigma_pct * price  # $/share/sqrt(day)

notional = 50_000.0  # $ to trade
X = notional / price  # shares

T = 1.0  # trading horizon, days
N = 13  # trading steps: half-hour buckets over a 6.5-hour session
tau = T / N  # step length, days
eta = 2.5e-6  # temporary impact coefficient, $*day/share^2
gamma = 2.5e-7  # permanent impact coefficient, $/share^2

# lambda: risk aversion, 1/$; kappa: decay rate, 1/day
lambdas_illustrative = {"risk_neutral": 0.0, "moderate": 4e-5, "aggressive": 2.5e-4}
kappas_illustrative = {
    label: decay_parameter(lam, sigma, eta, gamma, tau)
    for label, lam in lambdas_illustrative.items()
}
labels_display = {
    "risk_neutral": "Risk Neutral",
    "moderate": "Moderate",
    "aggressive": "Aggressive",
}

print(
    f"notional: ${notional:,.0f}, shares: {X:,.0f}, sigma: ${sigma:.2f}/share/sqrt(day)"
)
print("kappa by risk-aversion level:")
for label, kappa in kappas_illustrative.items():
    print(f"  {label}: {kappa:.3g}")

notional: $50,000, shares: 1,000, sigma: $0.50/share/sqrt(day)
kappa by risk-aversion level:
  risk_neutral: 0
  moderate: 2
  aggressive: 4.98


## Scheduled trading

The Almgren-Chriss closed-form schedule, $x_j = X \cdot \sinh(\kappa(T-t_j)) /
\sinh(\kappa T)$ at $t_j = j\tau$, for the three risk-aversion levels above:
risk-neutral ($\lambda=0$, a straight line), moderate, and aggressive.

In [4]:
t_fraction = np.arange(N + 1) / N
colors = {"risk_neutral": "#1f77b4", "moderate": "#ff7f0e", "aggressive": "#2ca02c"}

trajectories_df = pl.DataFrame(
    {
        "t_fraction": t_fraction,
        **{
            label: optimal_holdings(X, T, N, kappa)
            for label, kappa in kappas_illustrative.items()
        },
    }
)

fig = go.Figure()
for label, kappa in kappas_illustrative.items():
    legend_suffix = "λ=0" if kappa == 0.0 else f"κ={kappa:.2g}"
    fig.add_trace(
        go.Scatter(
            x=trajectories_df["t_fraction"],
            y=trajectories_df[label],
            mode="lines+markers",
            name=f"{labels_display[label]} ({legend_suffix})",
            line={"width": 2, "color": colors[label]},
        )
    )
fig.update_layout(
    title="Optimal Holdings Trajectory x_j",
    xaxis_title="Fraction of Time Elapsed",
    yaxis_title="Shares Held",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 0.5, "yanchor": "middle"},
)
fig.write_image("output/optimal_holdings_trajectory.png", scale=2)
fig.show()

## Changing speed mid-day

The general schedule runs between any two endpoints,
$x_j = (X_0 \sinh(\kappa(T - t_j)) + X_T \sinh(\kappa t_j)) / \sinh(\kappa T)$,
so a trader can re-plan partway through: take the shares still held as the new
$X_0$, the time left as the new $T$, and solve again, possibly with a
different $\lambda$. Here the trader starts on the moderate schedule and, after
6 of the 13 half-hour steps (12:30), re-plans the rest of the day at each of the
three risk-aversion levels.

Re-planning at the same $\lambda$ reproduces the original schedule exactly: the
optimal schedule is time-consistent, as Almgren and Chriss show, because nothing
learned along the way changes the forecast of future prices. A change of pace
therefore only comes from a change of $\lambda$. It shows up as a kink in the
holdings and a jump in the shares traded per step.

In [5]:
k_switch = 6  # steps completed before re-planning (12:30)

morning = optimal_holdings(X, T, N, kappas_illustrative["moderate"])[: k_switch + 1]

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Shares Held", "Shares Traded per Step"),
)
for label, kappa in kappas_illustrative.items():
    afternoon = optimal_holdings(morning[-1], T - k_switch * tau, N - k_switch, kappa)
    holdings = np.concatenate([morning, afternoon[1:]])
    trades = -np.diff(holdings)

    name = f"Moderate → {labels_display[label]}"
    fig.add_trace(
        go.Scatter(
            x=t_fraction,
            y=holdings,
            mode="lines+markers",
            name=name,
            legendgroup=label,
            line={"width": 2, "color": colors[label]},
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        # n_j is traded over (t_(j-1), t_j], so draw it as a step starting at
        # t_(j-1), repeating the last trade to close the final step
        go.Scatter(
            x=t_fraction,
            y=np.append(trades, trades[-1]),
            mode="lines",
            name=name,
            legendgroup=label,
            showlegend=False,
            line={"width": 2, "color": colors[label], "shape": "hv"},
        ),
        row=2,
        col=1,
    )
fig.add_vline(x=k_switch / N, line={"width": 1, "dash": "dot", "color": "gray"})
fig.update_xaxes(title_text="Fraction of Time Elapsed", row=2, col=1)
fig.update_layout(
    title="Re-planning at 12:30",
    width=700,
    height=600,
    margin={"l": 60, "r": 20, "t": 70, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 1, "yanchor": "top"},
)
fig.write_image("output/midday_replan.png", scale=2)
fig.show()

## Cost

Expected cost splits into a permanent-impact piece, $\gamma \sum_k n_k x_k =
\frac{1}{2}\gamma X^2 - \frac{1}{2}\gamma \sum_k n_k^2$, and a temporary-impact
piece, $(\eta / \tau) \sum_k n_k^2$. Both depend on the schedule only through
$\sum_k n_k^2$, which shrinks as trading is spread more evenly over the
horizon (lower risk aversion, lower $\kappa$).

How that cost accrues over the trading horizon: the cost accrued through
step $k$ is the same pair of sums truncated at $k$, $\gamma \sum_{i \le k} n_i
x_i + (\eta / \tau) \sum_{i \le k} n_i^2$, so it is evaluated by passing the
holdings $x_0, \dots, x_k$ to `expected_cost`.

In [6]:
fig = go.Figure()
for label, kappa in kappas_illustrative.items():
    holdings = optimal_holdings(X, T, N, kappa)
    cumulative_cost = [
        expected_cost(holdings[: k + 1], tau, gamma, eta) for k in range(N + 1)
    ]

    legend_suffix = "λ=0" if kappa == 0.0 else f"κ={kappa:.2g}"
    fig.add_trace(
        go.Scatter(
            x=t_fraction,
            y=cumulative_cost,
            mode="lines+markers",
            name=f"{labels_display[label]} ({legend_suffix})",
            line={"width": 2, "color": colors[label]},
        )
    )
fig.update_layout(
    title="Cumulative Expected Cost Over the Trading Horizon",
    xaxis_title="Fraction of Time Elapsed",
    yaxis_title="Cumulative Cost ($)",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 0.02, "y": 0.98},
)
fig.show()

## Efficient frontier

Sweeping the risk-aversion parameter $\lambda$ traces out the efficient
frontier: the lowest achievable expected cost for each level of risk. Risk is
shown as the standard deviation of cost (in $, the same units as cost itself)
rather than variance ($²), which is otherwise hard to interpret at a glance.
The three risk-aversion levels above are highlighted on the curve.

In [7]:
lambdas = np.concatenate([[0.0], np.logspace(-10, -2, 49)])
frontier = efficient_frontier(lambdas, X, T, N, sigma, eta, gamma)
frontier_df = pl.DataFrame(
    {
        "variance": frontier["variance"],
        "expected_cost": frontier["expected_cost"],
    }
).sort("variance")
frontier_df = frontier_df.with_columns(cost_std=frontier_df["variance"].sqrt())

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=frontier_df["cost_std"],
        y=frontier_df["expected_cost"],
        mode="lines",
        name="Efficient Frontier",
        line={"width": 2},
    )
)
for label, kappa in kappas_illustrative.items():
    holdings = optimal_holdings(X, T, N, kappa)
    fig.add_trace(
        go.Scatter(
            x=[np.sqrt(cost_variance(holdings, tau, sigma))],
            y=[expected_cost(holdings, tau, gamma, eta)],
            mode="markers",
            name=labels_display[label],
            marker={"size": 10, "color": colors[label]},
        )
    )
fig.update_layout(
    title="Almgren-Chriss Efficient Frontier",
    xaxis_title="Std. Deviation of Cost ($)",
    yaxis_title="Cost ($)",
    width=700,
    height=450,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 1, "yanchor": "top"},
)
fig.show()

## Replicating Figure 1

Almgren and Chriss plot the frontier with the Table 1 parameters: one million
shares at \$50, sold over $T = 5$ days in $N = 5$ daily trades. Their curve
also continues past the risk-neutral point B into $\lambda < 0$ (dashed): a
risk-seeking trader who delays selling pays more in both expected cost and
variance, so this branch is not efficient. For $\lambda < 0$, $\tilde\kappa^2 <
0$ and the schedule becomes $x_j = X \sin(\omega(T - t_j)) / \sin(\omega T)$,
with $\frac{2}{\tau^2}(1 - \cos(\omega\tau)) = -\tilde\kappa^2$. The branch
ends where the first trade would turn into a purchase,
$\omega(T - \tau/2) = \pi/2$, since the model assumes a pure sell program.

The fixed cost $\epsilon X$ (half the spread on every share) is the same for
every monotone schedule, so it is added on top of `expected_cost`. The
straight line is tangent at $\lambda = 10^{-6}$, with slope $-\lambda$, and
points A, B, C are $\lambda = 2 \times 10^{-6}$, $0$, and $-2 \times 10^{-7}$.

In [8]:
# Table 1 parameters, suffixed so the inputs above are not overwritten
X_paper = 1e6  # shares
T_paper = 5.0  # days
N_paper = 5  # daily trades
tau_paper = T_paper / N_paper  # days
sigma_paper = 0.95  # $/share/sqrt(day)
epsilon_paper = 0.0625  # $/share, half the bid-ask spread
eta_paper = 2.5e-6  # $*day/share^2
gamma_paper = 2.5e-7  # $/share^2

# Most negative lambda before the first trade turns into a purchase:
# omega(T - tau/2) = pi/2, mapped back through (2/tau) sin(omega tau/2)
eta_tilde_paper = eta_paper - 0.5 * gamma_paper * tau_paper
omega_max = np.pi / (2 * (T_paper - 0.5 * tau_paper))
lam_min = -(
    (2 / tau_paper * np.sin(0.5 * omega_max * tau_paper)) ** 2
    * eta_tilde_paper
    / sigma_paper**2
)


def frontier_point(lam: float) -> tuple[float, float]:
    kappa = decay_parameter(lam, sigma_paper, eta_paper, gamma_paper, tau_paper)
    holdings = optimal_holdings(X_paper, T_paper, N_paper, kappa)
    variance = cost_variance(holdings, tau_paper, sigma_paper)
    cost = (
        expected_cost(holdings, tau_paper, gamma_paper, eta_paper)
        + epsilon_paper * X_paper
    )
    return variance, cost


efficient_lambdas = np.concatenate([[0.0], np.logspace(-9, -4, 80)])
risk_seeking_lambdas = np.linspace(0.999 * lam_min, 0.0, 60)
efficient_points = np.array([frontier_point(lam) for lam in efficient_lambdas])
risk_seeking_points = np.array([frontier_point(lam) for lam in risk_seeking_lambdas])

# Attainable region: everything above the lower boundary traced by both branches
boundary = np.concatenate([efficient_points, risk_seeking_points])
boundary = boundary[np.argsort(boundary[:, 0])]
v_max, e_top = 2.1e12, 2.5e6

lam_tangent = 1e-6
v_tangent, e_tangent = frontier_point(lam_tangent)
v_line = np.array([0.0, v_max])

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=[*boundary[:, 0], v_max, 0.0],
        y=[*boundary[:, 1], e_top, e_top],
        fill="toself",
        fillcolor="rgba(0, 0, 0, 0.15)",
        line={"width": 0},
        name="Attainable",
        hoverinfo="skip",
    )
)
fig.add_trace(
    go.Scatter(
        x=efficient_points[:, 0],
        y=efficient_points[:, 1],
        mode="lines",
        name="Efficient frontier (λ ≥ 0)",
        line={"width": 2, "color": "black"},
    )
)
fig.add_trace(
    go.Scatter(
        x=risk_seeking_points[:, 0],
        y=risk_seeking_points[:, 1],
        mode="lines",
        name="Risk seeking (λ < 0)",
        line={"width": 2, "color": "black", "dash": "dash"},
    )
)
fig.add_trace(
    go.Scatter(
        x=v_line,
        y=e_tangent - lam_tangent * (v_line - v_tangent),
        mode="lines",
        name=f"Tangent, λ = {lam_tangent:.0e}",
        line={"width": 1, "color": "black"},
    )
)
fig.add_trace(
    go.Scatter(
        x=[v_tangent],
        y=[e_tangent],
        mode="markers",
        showlegend=False,
        marker={"size": 9, "color": "white", "line": {"width": 1, "color": "black"}},
    )
)
points_labeled = {"A": 2e-6, "B": 0.0, "C": -2e-7}
labeled = np.array([frontier_point(lam) for lam in points_labeled.values()])
fig.add_trace(
    go.Scatter(
        x=labeled[:, 0],
        y=labeled[:, 1],
        mode="markers+text",
        text=list(points_labeled),
        textposition="bottom center",
        showlegend=False,
        marker={
            "size": 9,
            "symbol": "square",
            "color": "white",
            "line": {"width": 1, "color": "black"},
        },
    )
)
fig.update_layout(
    title="Almgren-Chriss Figure 1: Efficient Frontier",
    xaxis_title="Variance V[x] ($²)",
    yaxis_title="Expected loss E[x] ($)",
    xaxis_range=[0.0, v_max],
    yaxis_range=[0.2e6, e_top],
    width=700,
    height=500,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 1, "yanchor": "top"},
)
fig.write_image("output/efficient_frontier.png", scale=2)
fig.show()

for name, (v, e) in zip(points_labeled, labeled):
    print(f"{name}: V = {v / 1e12:.3f}e12 $², E = {e / 1e6:.3f}e6 $")

A: V = 0.202e12 $², E = 1.141e6 $
B: V = 1.083e12 $², E = 0.662e6 $
C: V = 1.584e12 $², E = 0.718e6 $
